# Set-4 Hotel Booking & Occupancy Analytics: Q1 Complete Solution
**Run all:** `Kernel -> Restart & Run All`. Needs `Hotel_Data.xlsx` (sheets: Hotels, Rooms, Bookings) in the same folder.

## 0. Load the data, remove duplicates, join the tables

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

hotels   = pd.read_excel('Hotel_Data.xlsx', sheet_name='Hotels')
rooms    = pd.read_excel('Hotel_Data.xlsx', sheet_name='Rooms')
bookings = pd.read_excel('Hotel_Data.xlsx', sheet_name='Bookings')

print('Shapes (rows, cols):', hotels.shape, rooms.shape, bookings.shape)
for d in (hotels, rooms, bookings):
    display(d.head(3))
    print(d.isna().sum(), '\n')
print('Duplicate rows:', hotels.duplicated().sum(), rooms.duplicated().sum(), bookings.duplicated().sum())

In [ ]:
# duplicates: one row per ID
hotels   = hotels.drop_duplicates(subset=['Hotel_ID'])
rooms    = rooms.drop_duplicates(subset=['Room_ID'])
bookings = bookings.drop_duplicates(subset=['Booking_ID'])

# bring room and hotel details onto every booking (needed for capacity, Room_Status, Hotel_Name, City)
df = bookings.merge(rooms, on='Room_ID', how='left').merge(hotels, on='Hotel_ID', how='left')
print(df.shape)
print('Bookings with no matching room:', df['Room_Type'].isna().sum())

## 1. Categorical data (Room_Type, Meal_Plan, Booking_Channel)
Same idea as before: look, strip spaces + Title Case, replace leftovers, check.

In [ ]:
print(df['Room_Type'].value_counts(), '\n')
print(df['Meal_Plan'].value_counts(), '\n')
print(df['Booking_Channel'].value_counts())

In [ ]:
for col in ['Room_Type', 'Meal_Plan', 'Booking_Channel', 'Booking_Status', 'Room_Status', 'Guest_Type']:
    df[col] = df[col].str.strip().str.title()

df['Booking_Channel'] = df['Booking_Channel'].replace({'Ota': 'OTA', 'Walk In': 'Walk-in', 'Walk-In': 'Walk-in',
                                                       'Mobile-App': 'Mobile App'})

print(df['Room_Type'].value_counts(), '\n')
print(df['Meal_Plan'].value_counts(), '\n')
print(df['Booking_Channel'].value_counts())

## 2. Filtering and selection (Booking_Status, Room_Status, Guest_Type)
Keep only records that represent real stays:
- Booking_Status: keep **Confirmed** and **Completed** (cancelled / no-show bookings earn nothing)
- Room_Status: drop rooms **under Maintenance**
- Guest_Type: drop **Staff** (not paying guests)

In [ ]:
print('Before:', df.shape)
print(df['Booking_Status'].value_counts(), '\n')
print(df['Room_Status'].value_counts(), '\n')
print(df['Guest_Type'].value_counts())

In [ ]:
df = df[df['Booking_Status'].isin(['Confirmed', 'Completed'])]
df = df[df['Room_Status'] != 'Maintenance']
df = df[df['Guest_Type'] != 'Staff']

print('After :', df.shape)
print(df['Booking_Status'].unique(), df['Room_Status'].unique(), df['Guest_Type'].unique())

## 3. Guests_Count vs Room_Capacity vs Occupancy_Nights
A reasonable stay has: at least 1 guest, guests **not above room capacity**, and 1 to 30 nights.

In [ ]:
# how many unreasonable records?
print('Guests < 1            :', (df['Guests_Count'] < 1).sum())
print('Guests > Room_Capacity:', (df['Guests_Count'] > df['Room_Capacity']).sum())
print('Nights < 1 or > 30    :', ((df['Occupancy_Nights'] < 1) | (df['Occupancy_Nights'] > 30)).sum())

In [ ]:
valid = (
    (df['Guests_Count'] >= 1) &
    (df['Guests_Count'] <= df['Room_Capacity']) &
    (df['Occupancy_Nights'] >= 1) &
    (df['Occupancy_Nights'] <= 30)
)
df = df[valid].copy()

print(df.shape)
print(df[['Guests_Count', 'Room_Capacity', 'Occupancy_Nights']].describe())

## 4. Booking charges (Room_Rate, Discount, Gross_Amount, Final_Amount)
Rules:
- `Gross_Amount = Room_Rate x Occupancy_Nights`
- `Final_Amount = Gross_Amount - Discount%` , i.e. `Gross x (1 - Discount/100)`

In [ ]:
print(df[['Room_Rate', 'Discount', 'Gross_Amount', 'Final_Amount']].describe())

In [ ]:
# Room_Rate must be positive -> otherwise use the median rate of that Room_Type
df.loc[df['Room_Rate'] <= 0, 'Room_Rate'] = np.nan
df['Room_Rate'] = df['Room_Rate'].fillna(df.groupby('Room_Type')['Room_Rate'].transform('median'))

# Discount is a percentage: valid 0 to 100, else no discount
df.loc[(df['Discount'] < 0) | (df['Discount'] > 100), 'Discount'] = 0

# how many wrong amounts?
gross_ok = df['Room_Rate'] * df['Occupancy_Nights']
final_ok = gross_ok * (1 - df['Discount'] / 100)
print('Wrong Gross_Amount:', (~np.isclose(df['Gross_Amount'], gross_ok)).sum())
print('Wrong Final_Amount:', (~np.isclose(df['Final_Amount'], final_ok)).sum())

# recalculate
df['Gross_Amount'] = gross_ok
df['Final_Amount'] = final_ok.round(2)
print('After fix, wrong Final_Amount:', (~np.isclose(df['Final_Amount'], df['Gross_Amount'] * (1 - df['Discount'] / 100))).sum())

## 4b. Dates and Guest_Rating (needed for the dashboard)

In [ ]:
df['Booking_Date'] = pd.to_datetime(df['Booking_Date'])
df['Booking_Month'] = df['Booking_Date'].dt.month_name()

# Guest_Rating: valid 1 to 5, otherwise median
df.loc[(df['Guest_Rating'] < 1) | (df['Guest_Rating'] > 5), 'Guest_Rating'] = np.nan
df['Guest_Rating'] = df['Guest_Rating'].fillna(df['Guest_Rating'].median())

print(df.isna().sum().sum(), 'missing values left')
print(df['Guest_Rating'].describe())

## 5. Validation graphs (occupancy, room performance, guest rating)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 10))

# occupancy
ax[0,0].hist(df['Occupancy_Nights'], bins=range(1, 12))
ax[0,0].set_title('Occupancy Nights (stay length)')
df.groupby('City')['Occupancy_Nights'].sum().sort_values().plot(kind='barh', ax=ax[0,1])
ax[0,1].set_title('Total Nights Booked by City')
ax[0,2].scatter(df['Room_Capacity'], df['Guests_Count'], alpha=0.3)
ax[0,2].plot([0, 5], [0, 5], 'r--'); ax[0,2].set_title('Guests vs Capacity (no points above the line)')

# room performance
df.groupby('Room_Type')['Final_Amount'].sum().plot(kind='bar', ax=ax[1,0])
ax[1,0].set_title('Revenue by Room Type')
df.boxplot(column='Room_Rate', by='Room_Type', ax=ax[1,1])
ax[1,1].set_title('Room Rate by Room Type'); ax[1,1].figure.suptitle('')

# guest rating
ax[1,2].hist(df['Guest_Rating'], bins=10)
ax[1,2].set_title('Guest Rating Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# extra: average rating by room type and monthly bookings trend
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df.groupby('Room_Type')['Guest_Rating'].mean().plot(kind='bar', ax=ax[0], ylim=(0, 5))
ax[0].set_title('Average Guest Rating by Room Type')
df.groupby(df['Booking_Date'].dt.to_period('M')).size().plot(kind='line', marker='o', ax=ax[1])
ax[1].set_title('Bookings per Month')
plt.tight_layout(); plt.show()

## 6. Export processed CSVs
`hotel_bookings_clean.csv` is one flat table (booking + room + hotel details), so Power BI needs **no relationships**.

In [ ]:
df.to_csv('hotel_bookings_clean.csv', index=False)
hotels.to_csv('hotels_clean.csv', index=False)
rooms['Room_Type'] = rooms['Room_Type'].str.strip().str.title()
rooms['Room_Status'] = rooms['Room_Status'].str.strip().str.title()
rooms.to_csv('rooms_clean.csv', index=False)

print('Saved:', df.shape, hotels.shape, rooms.shape)
print('Booking_ID unique?', df['Booking_ID'].is_unique)

## Power BI DAX measures (Q2) - load `hotel_bookings_clean.csv`
```
Total Bookings = COUNTROWS(hotel_bookings_clean)
Total Guests = SUM(hotel_bookings_clean[Guests_Count])
Total Revenue = SUM(hotel_bookings_clean[Final_Amount])
Average Stay Duration = AVERAGE(hotel_bookings_clean[Occupancy_Nights])
Average Guest Rating = AVERAGE(hotel_bookings_clean[Guest_Rating])
```